# Fine-tune a small LLM for banking support (Colab)

Runtime → Change runtime type → **T4 GPU**. Then run the cells in order.

1. Install and get the code
2. Supervised fine-tuning with LoRA
3. Evaluate on the shared 300-message test set
4. Optional: DPO preference tuning, then evaluate again
5. Publish to the Hugging Face Hub

In [ ]:
!git clone https://github.com/<your-username>/banking-intent-finetune.git
%cd banking-intent-finetune
!pip install -q -r requirements.txt
import os, sys
sys.path.insert(0, 'src')

## 1. Quick trial run
Trains on 500 examples for a few minutes to check everything works before the full run.

In [ ]:
from bift import train_sft
train_sft.main(['--out', 'runs/trial', '--max-train', '500', '--epochs', '1'])

## 2. Full SFT run

In [ ]:
train_sft.main(['--out', 'runs/sft', '--epochs', '2'])

## 3. Loss curves

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
log = pd.DataFrame(json.load(open('runs/sft/training.json'))['log_history'])
ax = log.dropna(subset=['loss']).plot(x='step', y='loss', label='train loss')
log.dropna(subset=['eval_loss']).plot(x='step', y='eval_loss', ax=ax, label='validation loss')
plt.savefig('runs/sft/loss.png', dpi=150, bbox_inches='tight')

## 4. Evaluate and compare
Upload `predictions.csv` from the LLM Evaluation Harness (results/techniques/) to compare with the best prompt. Set your GPU price per hour to compute cost.

In [ ]:
from bift import evaluate
evaluate.main(['--runs', 'runs/sft', '--prompted', 'predictions.csv',
               '--gpu-dollars-per-hour', '<your GPU price>', '--prompted-cost-per-1000', '<from the harness report>'])

## 5. Optional: DPO preference tuning

In [ ]:
from bift import train_dpo
train_dpo.main(['--sft', 'runs/sft', '--out', 'runs/dpo'])
evaluate.main(['--runs', 'runs/sft', 'runs/dpo', '--prompted', 'predictions.csv'])

## 6. Publish the adapter and model card
Create a write token at huggingface.co/settings/tokens first.

In [ ]:
from huggingface_hub import login
login()
from peft import PeftModel
from transformers import AutoModelForCausalLM
base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
PeftModel.from_pretrained(base, 'runs/sft/adapter').push_to_hub('<your-username>/banking77-qwen2.5-0.5b-lora')